<a href="https://colab.research.google.com/github/Armilsyam/Transformers/blob/main/Project_transformers_import_BertModel%2C_BertTokenizer_MUH_ARMIL_SYAM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
from torch import nn
from transformers import BertModel, BertTokenizer
import os

# ==========================================
# BAGIAN 1: DEFINISI ARSITEKTUR CUSTOM
# (Berdasarkan Slide: "5. Ujicoba Sentiment")
# ==========================================
class BertForSentimentAnalysis(nn.Module):
    def __init__(self, bert_model):
        super(BertForSentimentAnalysis, self).__init__()
        # Menyimpan model pre-trained BERT
        self.bert = bert_model
        # Membuat layer klasifikasi linear (Input 768 dari dimensi hidden BERT, Output 3 Kelas)
        self.classifier = nn.Linear(768, 3)

    def forward(self, input_ids, attention_mask):
        # Forward pass melalui BERT
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)

        # Mengambil token [CLS] yang berada di indeks ke-0 untuk representasi seluruh kalimat
        cls_token = outputs.last_hidden_state[:, 0, :]

        # Memasukkan representasi [CLS] ke layer klasifikasi
        logits = self.classifier(cls_token)
        return logits

# ==========================================
# BAGIAN 2: FUNGSI UTAMA (MAIN PROGRAM)
# ==========================================
def jalankan_proyek():
    print("Memuat Pre-trained Model & Tokenizer...")
    # Menggunakan model multilingual agar paham bahasa Indonesia
    nama_model = 'bert-base-multilingual-cased'
    tokenizer = BertTokenizer.from_pretrained(nama_model)
    model_dasar = BertModel.from_pretrained(nama_model)

    # Inisialisasi model sentimen custom kita
    sentiment_model = BertForSentimentAnalysis(model_dasar)

    # --- UJI COBA INFERENSI ---
    # Kalimat uji coba berdasarkan slide
    teks_uji = "Belajar tentang Transformers itu menarik!"
    print(f"\nMelakukan Tokenisasi pada teks: '{teks_uji}'")

    # Proses tokenisasi menjadi tensor PyTorch
    inputs = tokenizer(teks_uji, return_tensors="pt")

    print("\nMenjalankan Prediksi (Forward Pass)...")
    # Mematikan perhitungan gradien karena kita hanya menguji (bukan melatih)
    with torch.no_grad():
        logits = sentiment_model(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask']
        )

    print("\nHasil Logits (Skor Mentah untuk 3 Kelas):")
    print(logits)

    # --- MENYIMPAN MODEL ---
    print("\nMenyimpan Model dan Tokenizer ke folder 'saved_model'...")
    if not os.path.exists("saved_model"):
        os.makedirs("saved_model")

    # Menyimpan tokenizer (Sesuai slide)
    tokenizer.save_pretrained("saved_model")
    # Karena kita memakai nn.Module PyTorch murni, kita simpan bobotnya (state_dict)
    torch.save(sentiment_model.state_dict(), "saved_model/pytorch_model.bin")
    print("Berhasil disimpan!")

if __name__ == "__main__":
    jalankan_proyek()

Memuat Pre-trained Model & Tokenizer...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Melakukan Tokenisasi pada teks: 'Belajar tentang Transformers itu menarik!'

Menjalankan Prediksi (Forward Pass)...

Hasil Logits (Skor Mentah untuk 3 Kelas):
tensor([[-0.5206, -0.1380,  0.0765]])

Menyimpan Model dan Tokenizer ke folder 'saved_model'...
Berhasil disimpan!


In [6]:
from sklearn.model_selection import KFold
import pandas as pd

# Asumsi Anda punya dataset df
df = pd.read_csv("dataset_sentimen.csv")

# Membagi data menjadi 5 bagian lipatan (folds)
kf = KFold(n_splits=5)

for train_index, val_index in kf.split(df.index):
    train_df = df.iloc[train_index]
    val_df = df.iloc[val_index]

    print(f"Melatih model pada {len(train_df)} data...")
    # --- Disini Anda memasukkan script proses training (looping epochs, optimasi loss) ---

    print("Evaluasi pada data validasi...")
    # val_loss, predictions, true_vals = evaluate(sentiment_model)
    # val_accuracy = accuracy_score_func(predictions, true_vals)

Melatih model pada 8 data...
Evaluasi pada data validasi...
Melatih model pada 8 data...
Evaluasi pada data validasi...
Melatih model pada 8 data...
Evaluasi pada data validasi...
Melatih model pada 8 data...
Evaluasi pada data validasi...
Melatih model pada 8 data...
Evaluasi pada data validasi...


In [8]:
import pandas as pd

# Create a dummy dataset for sentiment analysis
data = {
    'text': [
        'This movie is great!',
        'I hate this product.',
        'It was an okay experience.',
        'Absolutely fantastic service.',
        'Terrible quality, very disappointed.',
        'Neutral feelings about this one.',
        'Highly recommend!',
        'Could be better.',
        'The worst ever.',
        'Pretty good for the price.'
    ],
    'label': [
        'positive',
        'negative',
        'neutral',
        'positive',
        'negative',
        'neutral',
        'positive',
        'negative',
        'negative',
        'positive'
    ]
}
df_dummy = pd.DataFrame(data)

# Save the dummy dataset to 'dataset_sentimen.csv'
df_dummy.to_csv('dataset_sentimen.csv', index=False)
print("Created dummy 'dataset_sentimen.csv' file.")

Created dummy 'dataset_sentimen.csv' file.
